# Solutions · Assessment · Module 04

Mark **Part A** and **Part C** against these. Part B marked itself; the values are printed below so you
can see where an answer went wrong rather than only that it did.

**Marking Part A:** one mark each, for the substance. Where an answer has two required halves the mark is
all-or-nothing.

In [ ]:
import hashlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# SYNTHETIC. One row per support ticket, 300 customers over 18 months.
def load_tickets():
    rng = np.random.default_rng(404)
    n_customers = 300
    difficulty = rng.gamma(2.0, 1.0, n_customers)
    rows = []
    for customer in range(n_customers):
        for _ in range(int(np.clip(rng.poisson(1.5 + 2.2 * difficulty[customer]) + 1, 1, 16))):
            month = int(rng.integers(1, 19))
            channel = rng.choice(["email", "phone", "chat", "portal", "social"],
                                 p=[.34, .24, .20, .16, .06])
            length = float(np.round(np.exp(rng.normal(5.0, 0.6)), 0))
            urgent = int(rng.random() < 0.18)
            hours = np.round(np.maximum(0.5,
                             2.0 + 3.0 * difficulty[customer] + 0.004 * length - 1.6 * urgent
                             + {"email": 1.2, "phone": -0.8, "chat": -1.1,
                                "portal": 0.4, "social": 2.0}[channel]
                             + 0.05 * month + rng.normal(0, 1.6)), 2)
            rows.append((customer + 1, month, channel, length, urgent, hours))
    tickets = pd.DataFrame(rows, columns=["customer_id", "month", "channel",
                                          "description_length", "urgent", "resolved_hours"])
    tickets = tickets.sort_values(["month", "customer_id"]).reset_index(drop=True)
    tickets.loc[rng.random(len(tickets)) < 0.04 + 0.40 * (tickets.description_length > 200),
                "description_length"] = np.nan
    tickets["tickets_so_far"] = tickets.groupby("customer_id").cumcount()
    tickets["customer_lifetime_tickets"] = tickets.groupby("customer_id").customer_id.transform("size")
    return tickets


EXPECTED = {
    "B1": "73c89ed29d81", "B2": "7250e9d1955e", "B3": "b48ce96ec049", "B4": "3ba9e8c0d467",
    "B5": "069928cc9477", "B6": "4edce39c9fce", "B7": "fb05176b4d0e", "B8": "e975e26e882b",
}


def check(task, answer):
    # Marks one Part B answer without revealing it. Counts: whole numbers. Everything else: 2 dp.
    task = task.upper()
    if task not in EXPECTED:
        print("unknown task:", task)
        return
    candidates = [answer] if isinstance(answer, (int, np.integer)) else [
        round(float(answer) + delta, 2) for delta in (-0.01, 0.0, 0.01)]
    for value in candidates:
        text = str(int(value)) if isinstance(answer, (int, np.integer)) else "%.2f" % value
        if hashlib.sha256((task + "|" + text).encode()).hexdigest()[:12] == EXPECTED[task]:
            print("%s  correct" % task)
            return
    print("%s  not yet - check your working, then try again" % task)


tickets = load_tickets()
print("loaded %d tickets, %d columns" % tickets.shape)
print(tickets.head(3).to_string(index=False))

## Part A · Recall

**A1.** **Unit of observation, target, prediction time, horizon, availability.** The one that decides
whether a column may be used is **availability**: would this value have been known at the moment the
prediction is made?

**A2.** `skill = (baseline error - model error) / baseline error`. **Negative skill means the model is
worse than a rule that costs nothing**, and should not ship.

**A3.** The **validation set** chooses between models, features and hyperparameters, and may be looked at
as often as you like - its score is optimistic by construction. The **test set** estimates the chosen
model's performance and is looked at **once**, at the end, after every choice is final.

**A4.** *Does the same entity produce more than one row?* and *will the model predict the future from the
past?* Answers: plain stratified split / `GroupKFold` / chronological or forward chaining / grouped **and**
forward.

**A5.** **Target, duplicate-or-group, temporal, preprocessing.** A `Pipeline` prevents only the **fourth**
- it refits every step inside each fold. It does nothing about a leaky feature, a repeated entity or a
future row.

**A6.** **Severity is proportional to how much information about the target the fitted step absorbed.** A
`StandardScaler` never sees `y` - it learns two numbers per column from `X` - so it is **not dangerous**;
04-05 measured it at +0.0005.

**A7.** Compare the target between rows where the value is **present** and rows where it is **missing**;
if the gap is large relative to its standard error, the missingness is informative. If it is, **impute and
add a binary indicator column**, and say in the write-up that the value itself cannot be recovered.

**A8.** Because a linear model fits **one coefficient** for an ordinal column, so it can only express
"the target rises steadily with the code number". If the codes are administrative, no such pattern exists
and one straight line through unordered groups is nearly flat. 04-06 measured the loss at 90% of the
column's value.

**A9.** It measures **the cross-validated score of the best candidate, on the folds that selected it**. It
does **not** measure what that model will do on new data - and, as 04-07 found, a gap between it and a
nested score may not be selection bias at all.

**A10.** **Seeds, code, data, environment, hardware.** A `random_state` fixes **level 1 only** - the same
answer twice, on this machine, with today's libraries.

## Part B · The values, and where each goes wrong

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["description_length", "urgent", "month", "tickets_so_far"]
FEATURES = NUMERIC + ["channel"]
hours = tickets.resolved_hours
folds = KFold(5, shuffle=True, random_state=0)


def preparer(columns=NUMERIC, categorical=True):
    steps = [("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                                   ("scale", StandardScaler())]), columns)]
    if categorical:
        steps.append(("categorical", OneHotEncoder(handle_unknown="ignore"), ["channel"]))
    return ColumnTransformer(steps)


def scored(pipeline, frame, **kwargs):
    return -cross_val_score(pipeline, frame, hours, scoring="neg_mean_absolute_error",
                            cv=kwargs.pop("cv", folds), **kwargs).mean()


pipeline = Pipeline([("prepare", preparer()), ("estimate", Ridge())])

print("B1  customers appearing more than once : %d of %d"
      % ((tickets.customer_id.value_counts() > 1).sum(), tickets.customer_id.nunique()))
print("B2  predicting the global median       : %.4f"
      % mean_absolute_error(hours, np.full(len(hours), hours.median())))
print("B3  ridge pipeline, KFold(5)           : %.4f" % scored(pipeline, tickets[FEATURES]))
print("B4  the same, GroupKFold by customer   : %.4f"
      % scored(pipeline, tickets[FEATURES], cv=GroupKFold(5), groups=tickets.customer_id))
solo = Pipeline([("prepare", preparer(["customer_lifetime_tickets"], categorical=False)),
                 ("estimate", Ridge())])
print("B5  customer_lifetime_tickets ALONE    : %.4f"
      % scored(solo, tickets[["customer_lifetime_tickets"]]))
print("B6  description_length missing         : %.2f%%" % (100 * tickets.description_length.isna().mean()))
print("B7  columns out of the ColumnTransformer: %d"
      % preparer().fit_transform(tickets[FEATURES]).shape[1])

seed_scores = []
for seed in range(30):
    train_rows, test_rows = train_test_split(np.arange(len(tickets)), test_size=0.3, random_state=seed)
    fitted = Pipeline([("prepare", preparer()), ("estimate", Ridge())]).fit(
        tickets[FEATURES].iloc[train_rows], hours.iloc[train_rows])
    seed_scores.append(mean_absolute_error(hours.iloc[test_rows],
                                           fitted.predict(tickets[FEATURES].iloc[test_rows])))
seed_scores = np.array(seed_scores)
print("B8  sd over 30 random splits           : %.4f   (mean %.4f, range %.4f-%.4f)"
      % (seed_scores.std(), seed_scores.mean(), seed_scores.min(), seed_scores.max()))

**Where each one goes wrong.**

- **B1** - counting distinct customers (300) rather than those appearing more than once (293). Only seven
  customers raised a single ticket, which is the number that makes B4 necessary.
- **B2** - using the mean (4.41) instead of the median. For absolute error the median is the better
  constant, which is 03-01.
- **B3** - forgetting `shuffle=True`, or scoring on the training rows. The tickets are sorted by month, so
  an unshuffled `KFold` is accidentally a chronological split and gives a different answer.
- **B4** - passing `groups=` to the wrong argument, or using `KFold` and expecting `GroupKFold`'s answer.
  Note `GroupKFold` does not take `shuffle`.
- **B5** - including the other features. The task is that column **alone**, and the point is what it
  achieves without help.
- **B6** - computing it over the whole frame rather than that column.
- **B7** - answering 5. Four numeric columns pass through, `channel` becomes five, so **9**.
- **B8** - reporting the standard error (sd / sqrt(30)) instead of the standard deviation, or the range.

## Part C · Judgement

### C1 (5 marks) · Read the picture

In [ ]:
print("thirty splits: mean %.4f, sd %.4f, min %.4f, max %.4f"
      % (seed_scores.mean(), seed_scores.std(), seed_scores.min(), seed_scores.max()))
print("the colleague's 3.34 is %.2f standard deviations below the mean"
      % ((seed_scores.mean() - 3.34) / seed_scores.std()))
print("it is the %s of the thirty" % ("best" if 3.34 <= seed_scores.min() + 0.01 else "not the best"))

**1. Where 3.34 sits.** At the extreme left of the distribution - it is essentially **the best of the
thirty splits**, about 2.9 standard deviations below the mean of 3.6062.

What to say: not that they are wrong, because 3.34 is a real number honestly computed on genuinely
held-out data. What to say is that **it is one draw from a distribution we can measure**, and it is the
luckiest one. The reportable figure is the mean with its spread. If they arrived at that seed by trying
several, it is 04-03's selection premium with seeds as the candidates; if they arrived at it by typing
`random_state=0` once, it is 04-08's accident. **Either way the fix is the same: report the distribution,
not a member of it.**

**2. The number to report alongside the mean: the standard deviation, 0.0912** - or equivalently a range.
Without it, a reader cannot tell whether a difference between two models is real. The whole
grouped-versus-random difference in this assessment is 0.01, about a ninth of this spread.

*Marks: 2 for locating 3.34 as an extreme, 2 for the spread and why, 1 for part 3.*

**3. Why B4 being worse is not evidence that grouping hurt.**

The two numbers differ by about **0.01**, and the split-to-split standard deviation is **0.0912** - nine
times larger. **The difference is well inside the noise of either measurement.**

There is also a second reason, which is the one this module keeps returning to: the two procedures differ
in more than one way. `GroupKFold` does not shuffle and produces folds of unequal size, so a difference
between it and a shuffled `KFold` mixes the grouping effect with fold-composition effects. To measure
grouping alone you would hold everything else fixed and compare on the same folds - 04-07's A/B/C/D table.

**And the deeper point: "grouping made the model worse" is a category error.** Grouping does not change
the model at all. It changes what the score is measuring - from "how well does it do on new tickets from
customers it knows" to "how well does it do on customers it has never seen". A lower number there is not
degradation; it is a more honest answer to a harder and more relevant question.

### C2 (5 marks) · The column that helps too much

In [ ]:
with_leak = Pipeline([("prepare", preparer(NUMERIC + ["customer_lifetime_tickets"])),
                      ("estimate", Ridge())])
print("with customer_lifetime_tickets : %.4f"
      % scored(with_leak, tickets[NUMERIC + ["customer_lifetime_tickets", "channel"]]))
print("without it                     : %.4f" % scored(pipeline, tickets[FEATURES]))
print("that column alone (B5)         : %.4f"
      % scored(solo, tickets[["customer_lifetime_tickets"]]))
print()
print("correlation of customer_lifetime_tickets with resolved_hours: %.4f"
      % tickets.customer_lifetime_tickets.corr(hours))

**1. The improvement is real and large: 3.0185 against 3.5837**, a 16% reduction in error from one column.

**2. Why it is not usable.** `customer_lifetime_tickets` is the **total** number of tickets that customer
will ever raise. For a ticket in month 3, that total includes tickets raised in months 4 to 18 - **which
have not happened yet.** Its value only becomes knowable once the customer's entire history is complete,
which is after every prediction it would be used for.

It is 04-01's `months_on_file` in a new costume, and it fails the availability question outright: at the
moment the prediction is made, this number does not exist.

**3. Why B5 is the strongest evidence.** That single column, alone, scores **3.2158** - better than all
five legitimate features together (**3.5837**). **A model with one feature should not beat a model with
five, unless that feature knows something the others cannot.**

That comparison is stronger than the improvement in part 1 because an improvement can always be argued to
be a genuinely good feature. "One column beats everything else combined" cannot: it is the signature of a
column that has seen the answer. It is 04-05's single-feature scan, and it is the check to run whenever a
new feature produces a surprising gain.

**4. A legitimate column from the same fact:** **`tickets_so_far`** - already in the table, and used in
the model - is the count of that customer's *previous* tickets, which is knowable at prediction time. Also
acceptable: tickets in the last 30/90 days, or the number raised before the start of the current month.
The distinction is not what the column measures but **which side of the prediction moment its inputs come
from**.

*Marks: 1 for the two numbers, 2 for the availability argument being specific about timing, 1 for the B5
comparison, 1 for a legitimate alternative.*

### C3 (4 marks) · Shipping it

**1. The splitting strategy: `GroupKFold` grouped by `customer_id`** - or `StratifiedGroupKFold` if the
target were categorical.

The deployment described is explicit: **new tickets from customers the company has not dealt with
before**. That is exactly the question a grouped split asks and a random split does not. A random split
would train on some of a customer's tickets and test on others, measuring performance on *familiar*
customers - which is not what was asked for.

Credit for also raising **time**: the model runs every morning on tickets that arrive after training, so a
chronological element is defensible. The strongest answer says grouping is required by the stated
deployment and chronology is worth adding, and notes that combining them costs data - 04-04's fourth
column.

**2. Which number is honest.** **B4, the grouped estimate**, because it is the only one that measures
performance on unseen customers. Reporting B3 instead would overstate performance by about **0.01 hours** -
which, and this is the part worth saying out loud, is **negligible here**: about a ninth of the
split-to-split spread, so on this dataset the choice of splitter barely changes the answer.

**That does not make the choice unimportant.** It makes it *measurably* unimportant on this data, which is
a conclusion you can only reach by computing both. 04-04's gym data gave +0.0957 for a random forest and a
reversed model ranking; the same code on this data gives 0.01. **The size of the effect is a property of
the data and the model, and the only way to know it is to measure it.**

**3. Three things that must be handed over** - any three of:

- the **serialised pipeline**, carrying its own imputer, scaler and encoder
- the **pinned library versions** it was fitted with, since unpickling across versions can fail silently
- the **expected input schema** - column names, dtypes and units, because a `ColumnTransformer` addresses
  by name
- the **evaluation record**: metric, value, **spread**, baseline and splitting rule, so "is it still
  working?" has a reference
- the **prediction contract** from 04-01 - unit, target, prediction time, horizon - so nobody calls it for
  a different question
- a **worked example**: one input row and its expected output, as a one-second smoke test

*Marks: 1 for `GroupKFold` with the reason, 1 for identifying B4 as honest, 1 for quantifying the
difference against the noise, 1 for three handover items.*